This file imports an EXCEL model (here template example from BW25 sharepoint folder) and matches it with the background database in the project. For this repository we are currently working with ecoinvent-3.12-consequential

After successfully importing a database to a Brightway project, the database is stored in the project and doesn't have to be imported every time we open the repository. Unless we make changes to the background database of course.

# EXCEL Model importer

## 1. Import packages

In [4]:
import bw2data as bd
import bw2io as bi
import pandas as pd
from functions.utils import Config

## 2. Set project and see databases
The databases are required here, because the Excel database we want to import in this script needs to be matched to these.

In [5]:
# define a project where we install the databases and work in this script
bd.projects.set_current(Config.PROJECT_NAME)

In [6]:
bd.databases

Databases dictionary with 2 object(s):
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-consequential

## 3. Import the EXCEL file containing the model 
The file import is easy, we just use the Excel importer function from bw2io as can be seen below.

In [9]:
# Use the project Model directory configured in functions/utils.py.
excel_model_path = Config.PROJECT_DIRECTORY_PATH/"BW_LCI_importer_template.xlsx"

In [11]:
# Change the workbook filename above if your model uses a different name.
brightway_database_template_importer = bi.ExcelImporter(excel_model_path)

Extracted 1 worksheets in 0.04 seconds


## 4. Importing means matching with background databases and writing the database
The part that is a bit trickier to understand is how the matching with the relevant background databases works. In essence, you need to match the newly imported databases with all background databases used in the model.

In [12]:
brightway_database_template_importer.apply_strategies()
brightway_database_template_importer.match_database(fields=['name','location'])
brightway_database_template_importer.match_database(db_name='ecoinvent-3.12-consequential', fields=['name','location','unit'])
brightway_database_template_importer.match_database(db_name='ecoinvent-3.12-biosphere', fields=['name','categories','unit'])

Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_restore_temporal_distributions
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 15 strategies in 3.36 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [13]:
# check foreground imports for unlinked processes
pd.DataFrame(brightway_database_template_importer.unlinked)

""


In [14]:
# if there are unlinked exchanges, this helps localizing them in order to fix them
list(brightway_database_template_importer.unlinked)

[]

In [16]:
brightway_database_template_importer.write_database()

13:53:39+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 1/1 [00:00<?, ?it/s]


13:53:39+0200 [info     ] Vacuuming database            
Created database: brightway_project_template


## 5. Store data in a dataframe so it can be analysed in Brightway 2.5
This step allows you to analyse the imported database in the Brightway framework.

In [17]:
LCI_template = bd.Database('brightway_database_template_importer')

In [18]:
bd.databases

Databases dictionary with 3 object(s):
	brightway_project_template
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-consequential

In [19]:
ei_bio = bd.Database('ecoinvent-3.12-biosphere')
ei_clca = bd.Database('ecoinvent-3.12-consequential')
template_consequential = bd.Database('template_consequential')